# BIFlow ? 05. Orchestration DAG, Int?gration Multi-Agents & Gouvernance

Ce notebook documente et ex?cute l'int?gration fonctionnelle des agents BIFlow :
- **Orchestrateur DAG de Samar** (`graphlib.TopologicalSorter`, SQLite task store)
- **Intake, Similarit? hongroise & Classification hybride de Malek**
- **Privacy & Cleaning d'Ikbel**
- **Profiling, ETL DuckDB & Lignage d'Ahmed**
- **Analyst, KPI Catalog & Compilateur SQL de Rabeb**
- **Comparisons Checker, Auditor & Journal XAI de Tasnim**
- **Dashboard de Samar**


## 1. Inventaire des Agents & D?finition du DAG
Nous inspectons le graphe de d?pendances officiel d?fini dans `contracts.pipeline.STEPS`.

In [ ]:
from contracts.pipeline import STEPS, execution_order, graph
import pandas as pd

order = execution_order()
print("Ordre d'ex?cution topologique valid? :")
for idx, step_name in enumerate(order, 1):
    step = STEPS[step_name]
    print(f"{idx:2d}. {step.name.value:<15} (Owner: {step.owner:<7}) -> Outputs: {list(step.outputs.keys())}")


## 2. Ex?cution du DAG avec l'Orchestrateur R?el
Nous ex?cutons le pipeline complet avec `orchestrator.dag_runner.run_pipeline`.

In [ ]:
from pathlib import Path
from orchestrator.dag_runner import run_pipeline, get_run_status
from contracts.agent import AgentStatus

RUN_ID = "nb_demo_001"
results = run_pipeline(
    run_id=RUN_ID,
    params={
        "source_dir": "data/sample",
        "question": "Revenue by state in August",
        "uploaded_by": "NotebookUser"
    },
    stop_on_failure=False
)

print(f"\nR?sultats d'ex?cution du run {RUN_ID} :")
for name, res in results.items():
    print(f"  [{res.status.value.upper():>12}] {name:<15}: {res.message}")


## 3. Inspection des Artefacts S?mantiques Produits
V?rification des sorties riches : `column_matches.json` (affectation hongroise) et `semantic_metadata.json` (classification 5 tiers).

In [ ]:
import json

meta_file = Path(f"runs/{RUN_ID}/column_types/semantic_metadata.json")
if meta_file.exists():
    meta = json.loads(meta_file.read_text(encoding="utf-8"))
    print(f"Colonnes classifi?es : {meta['total_columns']}")
    for col in meta['columns'][:5]:
        print(f"  - {col['column_name']} : {col['semantic_type']} (r?le: {col['semantic_role']}, pii: {col['privacy_hint']})")

matches_file = Path(f"runs/{RUN_ID}/similarity/column_matches.json")
if matches_file.exists():
    matches = json.loads(matches_file.read_text(encoding="utf-8"))
    print(f"\nCorrespondances inter-tables (m?thode: {matches['method']}) : {matches['matches_found']}")
    for m in matches['matches'][:5]:
        print(f"  - {m['source_column']} <-> {m['target_column']} (relation: {m['relationship']}, score: {m['similarity_score']:.2f})")


## 4. V?rification de la Tra?abilit?, du Lignage et de l'Audit
Contr?le de l'immuabilit? du journal cryptographique SHA-256.

In [ ]:
from contracts.audit import verify_journal, read_events, DEFAULT_JOURNAL

if DEFAULT_JOURNAL.exists():
    status = verify_journal(DEFAULT_JOURNAL)
    print(f"Int?grit? du journal audit : {'VALIDE' if status.ok else 'FALSIFI?'}")
    events = read_events(DEFAULT_JOURNAL)
    print(f"Nombre d'?v?nements audit?s : {len(events)}")
    for ev in events[-5:]:
        print(f"  [{ev.timestamp[:19]}] {ev.agent} -> {ev.action} ({ev.status})")
